In [ ]:
from pathlib import Path

# === RUN THIS FIRST: repo-root anchor so paths resolve from any working dir ===
# Walks up until it finds the repo root (the folder containing both ai/ and src/).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "ai").is_dir() and (p / "src").is_dir()), Path.cwd())

DATASETS = ROOT / "ai" / "data" / "datasets"                      # datasets root
RUNS     = ROOT / "ai" / "runs"                                   # training/eval output
DEPLOY   = ROOT / "ai" / "models" / "subsystem1" / "production"   # deployed .pt models

print("ROOT     =", ROOT)
print("DATASETS =", DATASETS)
print("RUNS     =", RUNS)
print("DEPLOY   =", DEPLOY)

In [3]:
import os
import glob
import cv2
import yaml
import random
import albumentations as A
from tqdm import tqdm
import hashlib

def advanced_trash_augmentation(dataset_dir, multiplication_factor=3):
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    train_img_dir = os.path.join(dataset_dir, "train", "images")
    train_lbl_dir = os.path.join(dataset_dir, "train", "labels")

    # Validate directories exist
    if not os.path.isdir(train_img_dir):
        print(f"Error: Training image directory not found -> {train_img_dir}")
        return
    if not os.path.isdir(train_lbl_dir):
        print(f"Error: Training label directory not found -> {train_lbl_dir}")
        return

    # Collect images and skip any already-augmented files from previous runs
    all_img_files = glob.glob(os.path.join(train_img_dir, "*.*"))
    img_files = [
        f for f in all_img_files
        if "_aug_" not in os.path.basename(f) and os.path.isfile(f)
    ]

    skipped = len(all_img_files) - len(img_files)
    if skipped > 0:
        print(f"Skipped {skipped} already-augmented files from a previous run.")

    if not img_files:
        print("Error: No original images found to augment.")
        return

    print(f"Found {len(img_files)} original training images.")
    print(f"Will generate {len(img_files) * multiplication_factor} augmented copies.")

    trash_pipeline = A.Compose([
        # 1. Flips and 90 degree rotations
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        
        # 2. Rotation and shear to mimic deformed cans
        A.Affine(
            rotate=(-45, 45),
            shear=(-10, 10),
            cval=(114, 114, 114),
            p=0.6
        ),
        
        # 3. Crop and zoom
        A.RandomResizedCrop(
            size=(640, 640),
            scale=(0.7, 1.0),
            p=0.5
        ),
        
        # 4. Grayscale for 15% of images
        A.ToGray(p=0.15),
        
        # 5. Color and brightness adjustments
        A.HueSaturationValue(
            hue_shift_limit=(-15, 15),
            sat_shift_limit=(-25, 25),
            val_shift_limit=(-15, 15),
            p=0.6
        ),
        A.RandomBrightnessContrast(
            brightness_limit=(-0.15, 0.15),
            contrast_limit=(-0.10, 0.10),
            p=0.6
        ),
        
        # 6. Blur and noise
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.GaussNoise(var_limit=(10.0, 30.0), p=0.2)

    ], bbox_params=A.BboxParams(
        format='yolo',
        label_fields=[],
        min_visibility=0.4,
        min_area=25
    ))

    success_count = 0
    skip_count = 0
    error_count = 0

    for img_path in tqdm(img_files, desc="Augmenting"):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        lbl_path = os.path.join(train_lbl_dir, f"{base_name}.txt")

        # Skip if label file is missing
        if not os.path.isfile(lbl_path):
            skip_count += 1
            continue

        image = cv2.imread(img_path)
        if image is None:
            print(f"  Warning: Could not read image -> {img_path}")
            skip_count += 1
            continue

        h, w = image.shape[:2]

        # Parse bounding boxes from label file
        bboxes = []
        with open(lbl_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    bboxes.append([
                        float(parts[1]),
                        float(parts[2]),
                        float(parts[3]),
                        float(parts[4]),
                        int(parts[0])
                    ])

        for i in range(multiplication_factor):
            # Skip if output files already exist from a partial run
            short_id = hashlib.md5(base_name.encode()).hexdigest()[:8]
            new_base_name = f"aug_{short_id}_v{i+1}"
            out_img_path = os.path.join(train_img_dir, f"{new_base_name}{ext}")
            out_lbl_path = os.path.join(train_lbl_dir, f"{new_base_name}.txt")

            if os.path.isfile(out_img_path) and os.path.isfile(out_lbl_path):
                skip_count += 1
                continue

            try:
                augmented = trash_pipeline(image=image, bboxes=bboxes)
                aug_img = augmented['image']
                aug_boxes = augmented['bboxes']

                # Skip if all bounding boxes were lost during augmentation
                if len(aug_boxes) == 0 and len(bboxes) > 0:
                    skip_count += 1
                    continue

                # Resize back to original dimensions
                aug_img = cv2.resize(aug_img, (w, h))

                # Write image
                write_ok = cv2.imwrite(out_img_path, aug_img)
                if not write_ok:
                    print(f"  Warning: Failed to write image -> {out_img_path}")
                    error_count += 1
                    continue

                # Write label
                with open(out_lbl_path, 'w') as f:
                    for box in aug_boxes:
                        f.write(f"{int(box[4])} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")

                success_count += 1

            except Exception as e:
                print(f"  Warning: Augmentation failed for {base_name} v{i+1} -> {e}")
                error_count += 1
                continue

    print(f"\nDone!")
    print(f"  Successfully created : {success_count} augmented pairs")
    print(f"  Skipped              : {skip_count} (missing labels, lost boxes, or already exist)")
    print(f"  Errors               : {error_count}")
    print(f"\nYour original images in '{train_img_dir}' were not modified.")


advanced_trash_augmentation(str(DATASETS / "tin.yolov8.split"), multiplication_factor=3)

Found 1033 original training images.
Will generate 3099 augmented copies.


Augmenting:   2%|▏         | 24/1033 [00:35<14:32,  1.16it/s] 

Augmenting:   5%|▍         | 51/1033 [01:25<12:17,  1.33it/s]  

Augmenting:   6%|▌         | 59/1033 [01:47<22:34,  1.39s/it]  

Augmenting:   6%|▌         | 62/1033 [01:49<14:13,  1.14it/s]

...

Augmenting:  99%|█████████▉| 1023/1033 [11:49<00:03,  2.57it/s]

Augmenting: 100%|█████████▉| 1029/1033 [11:49<00:00,  5.63it/s]

Augmenting: 100%|█████████▉| 1031/1033 [11:50<00:00,  4.99it/s]

Augmenting: 100%|██████████| 1033/1033 [11:50<00:00,  1.45it/s]


Done!
  Successfully created : 2337 augmented pairs
  Skipped              : 33 (missing labels, lost boxes, or already exist)
  Errors               : 729

Your original images in '../data/tin.yolov8.split\train\images' were not modified.
